# Módulo 1 · Clase 12 — Proyecto Integrador
# Sistema de Gestión de Biblioteca

**Curso de Inteligencia Artificial — Nivel básico**

---

## Bienvenida al proyecto integrador

Esta clase cierra el Módulo 1. Ya no vamos a aprender conceptos nuevos: vamos a **unir todo lo que sabes** para construir un sistema real de principio a fin.

Un proyecto integrador es diferente a los ejercicios anteriores:
- No hay una solución única ni un camino marcado.
- Tendrás que **tomar decisiones de diseño** y justificarlas.
- El objetivo es que al terminar seas capaz de leer el código y entender **por qué** está escrito así, no solo **qué** hace.

---

## Qué conceptos pone en práctica

| Clase | Concepto | Dónde aparece en este proyecto |
|-------|----------|--------------------------------|
| C1–C3 | Variables, tipos, operadores | En todas partes |
| C4–C5 | Control de flujo, bucles | Menú, búsquedas, filtros |
| C6 | Colecciones | Catálogo, préstamos, historial |
| C7 | Funciones | Búsqueda, informes, utilidades |
| C8 | Archivos (JSON) | Persistencia del catálogo y datos |
| C9 | Excepciones | Dominio propio: `MaterialNoDisponible`, etc. |
| C10–C11 | POO (herencia, polimorfismo, ABC) | `Material → Libro / Revista / DVD` |

---

## El sistema que vamos a construir

```
┌─────────────────────────────────────────────────┐
│              BIBLIOTECA DIGITAL                 │
├─────────────────┬───────────────────────────────┤
│   CATÁLOGO      │   USUARIOS      │   PRÉSTAMOS │
│  Libro          │   Usuario       │   Prestamo  │
│  Revista        │   (nombre,      │   (quién,   │
│  DVD            │    email,       │    qué,     │
│  (herencia ABC) │    historial)   │    cuándo)  │
├─────────────────┴───────────────────────────────┤
│              BIBLIOTECA (orquestador)           │
│  buscar · prestar · devolver · informes         │
├─────────────────────────────────────────────────┤
│              PERSISTENCIA (JSON)                │
│  guardar() · cargar()                           │
└─────────────────────────────────────────────────┘
```

## Plan de construcción

| Bloque | Qué construyes |
|--------|----------------|
| **1** | Excepciones y jerarquía de `Material` (ABC + subclases) |
| **2** | Clases `Usuario` y `Prestamo` |
| **3** | Clase `Biblioteca`: catálogo, alta y baja de materiales |
| **4** | Préstamos, devoluciones y búsqueda polimórfica |
| **5 (reto)** | Persistencia JSON, informes y menú interactivo |

---
## Bloque 1 — Excepciones y jerarquía de materiales

### 1.1 Excepciones del dominio

Antes de escribir ninguna clase de negocio, definimos las excepciones. Así el resto del código puede lanzarlas y capturarlas con nombres significativos.

In [ ]:
# ─── Jerarquía de excepciones de la biblioteca ────────────

class ErrorBiblioteca(Exception):
    """Base para todos los errores de la biblioteca."""

class MaterialNoEncontrado(ErrorBiblioteca):
    """El ISBN / ID buscado no existe en el catálogo."""

class MaterialNoDisponible(ErrorBiblioteca):
    """El material existe pero está prestado."""

class UsuarioNoEncontrado(ErrorBiblioteca):
    """El ID de usuario no existe."""

class LimitePrestamosAlcanzado(ErrorBiblioteca):
    """El usuario ya tiene el máximo de préstamos activos."""

class PrestamoNoEncontrado(ErrorBiblioteca):
    """No hay ningún préstamo activo para ese material y usuario."""

class DatosInvalidos(ErrorBiblioteca):
    """Datos de entrada incorrectos o incompletos."""

print("✅ Jerarquía de excepciones definida")

### 1.2 La jerarquía de materiales

Todos los materiales de la biblioteca comparten una interfaz común — la clase abstracta `Material`. Cada tipo concreto añade sus propios atributos y sobreescribe los métodos que lo diferencian.

```
Material (ABC)
  id, titulo, disponible
  prestar(), devolver()     ← concretos en la base
  tipo(), ficha()           ← abstractos, cada subclase los define
  to_dict() / from_dict()   ← para serializar a JSON
       │
       ├── Libro      (autor, isbn, genero, paginas)
       ├── Revista    (editorial, numero, mes_anio)
       └── DVD        (director, duracion_min, clasificacion)
```

**Enunciado bloque 1:** Implementa la clase `Material` y sus tres subclases. Requisitos:

- `Material.__init__` asigna un `id` autoincremental (`M001`, `M002`…), `titulo` y `disponible=True`.
- `prestar()` marca `disponible=False` o lanza `MaterialNoDisponible`.
- `devolver()` marca `disponible=True`.
- `@abstractmethod tipo()` → devuelve `"Libro"`, `"Revista"` o `"DVD"`.
- `@abstractmethod ficha()` → devuelve un string con los detalles del material.
- `to_dict()` → serializa a diccionario (para guardar en JSON). Incluye `"clase": self.tipo()`.
- `@classmethod from_dict(cls, datos)` → reconstruye el objeto desde un diccionario.
- `__str__` usa `ficha()`; `__repr__` incluye `tipo()`, `id` y `titulo`; `__eq__` compara por `id`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from abc import ABC, abstractmethod


<details>
<summary>💡 Ver solución — Bloque 1</summary>

```python
from abc import ABC, abstractmethod

class Material(ABC):
    _contador = 0

    def __init__(self, titulo):
        Material._contador += 1
        self.id          = f"M{Material._contador:03d}"
        self.titulo      = titulo.strip()
        self.disponible  = True

    # ─── Métodos concretos ───────────────────────────────
    def prestar(self):
        if not self.disponible:
            raise MaterialNoDisponible(
                f"'{self.titulo}' [{self.id}] ya está prestado"
            )
        self.disponible = False

    def devolver(self):
        self.disponible = True

    # ─── Métodos abstractos ──────────────────────────────
    @abstractmethod
    def tipo(self):
        pass

    @abstractmethod
    def ficha(self):
        pass

    # ─── Serialización ───────────────────────────────────
    def to_dict(self):
        return {
            "clase":      self.tipo(),
            "id":         self.id,
            "titulo":     self.titulo,
            "disponible": self.disponible,
        }

    @classmethod
    def from_dict(cls, datos):
        """Reconstruye un Material desde un diccionario. Ajustar en subclases."""
        obj = cls.__new__(cls)          # no llama a __init__
        obj.id         = datos["id"]
        obj.titulo     = datos["titulo"]
        obj.disponible = datos["disponible"]
        return obj

    # ─── Dunder ──────────────────────────────────────────
    def __str__(self):
        estado = "✅" if self.disponible else "🔴"
        return f"{estado} [{self.id}] {self.ficha()}"

    def __repr__(self):
        return f"{self.tipo()}(id={self.id!r}, titulo={self.titulo!r})"

    def __eq__(self, otro):
        if not isinstance(otro, Material):
            return NotImplemented
        return self.id == otro.id

    def __hash__(self):
        return hash(self.id)


class Libro(Material):
    def __init__(self, titulo, autor, isbn, genero="General", paginas=0):
        super().__init__(titulo)
        self.autor   = autor.strip()
        self.isbn    = isbn.strip()
        self.genero  = genero
        self.paginas = int(paginas)

    def tipo(self):
        return "Libro"

    def ficha(self):
        return (
            f"LIBRO  · {self.titulo} / {self.autor} "
            f"· ISBN {self.isbn} · {self.genero} · {self.paginas} págs."
        )

    def to_dict(self):
        d = super().to_dict()
        d.update({"autor": self.autor, "isbn": self.isbn,
                  "genero": self.genero, "paginas": self.paginas})
        return d

    @classmethod
    def from_dict(cls, datos):
        obj = cls(datos["titulo"], datos["autor"], datos["isbn"],
                  datos.get("genero", "General"), datos.get("paginas", 0))
        obj.id         = datos["id"]
        obj.disponible = datos["disponible"]
        return obj


class Revista(Material):
    def __init__(self, titulo, editorial, numero, mes_anio):
        super().__init__(titulo)
        self.editorial = editorial.strip()
        self.numero    = int(numero)
        self.mes_anio  = mes_anio.strip()    # ej: "Junio 2025"

    def tipo(self):
        return "Revista"

    def ficha(self):
        return (
            f"REVISTA· {self.titulo} · Ed. {self.editorial} "
            f"· Nº {self.numero} · {self.mes_anio}"
        )

    def to_dict(self):
        d = super().to_dict()
        d.update({"editorial": self.editorial, "numero": self.numero,
                  "mes_anio": self.mes_anio})
        return d

    @classmethod
    def from_dict(cls, datos):
        obj = cls(datos["titulo"], datos["editorial"],
                  datos["numero"], datos["mes_anio"])
        obj.id         = datos["id"]
        obj.disponible = datos["disponible"]
        return obj


class DVD(Material):
    def __init__(self, titulo, director, duracion_min, clasificacion="TP"):
        super().__init__(titulo)
        self.director     = director.strip()
        self.duracion_min = int(duracion_min)
        self.clasificacion = clasificacion.strip().upper()

    def tipo(self):
        return "DVD"

    def ficha(self):
        return (
            f"DVD    · {self.titulo} · Dir. {self.director} "
            f"· {self.duracion_min} min · {self.clasificacion}"
        )

    def to_dict(self):
        d = super().to_dict()
        d.update({"director": self.director, "duracion_min": self.duracion_min,
                  "clasificacion": self.clasificacion})
        return d

    @classmethod
    def from_dict(cls, datos):
        obj = cls(datos["titulo"], datos["director"],
                  datos["duracion_min"], datos.get("clasificacion", "TP"))
        obj.id         = datos["id"]
        obj.disponible = datos["disponible"]
        return obj


# ─── Pruebas ─────────────────────────────────────────────
Material._contador = 0

l1 = Libro("El Quijote",   "Miguel de Cervantes", "978-84-376-0494-7", "Clásico", 863)
l2 = Libro("Dune",         "Frank Herbert",       "978-84-450-7652-0", "Ciencia ficción", 604)
r1 = Revista("National Geographic", "National Geographic Society", 312, "Mayo 2025")
d1 = DVD("Blade Runner 2049", "Denis Villeneuve", 164, "16")

for m in [l1, l2, r1, d1]:
    print(m)

print()
l1.prestar()
print(f"Tras prestar: {l1}")

try:
    l1.prestar()
except MaterialNoDisponible as e:
    print(f"❌ {e}")

l1.devolver()
print(f"Tras devolver: {l1}")

print()
print("to_dict:", l1.to_dict())
l1_reconstruido = Libro.from_dict(l1.to_dict())
print("from_dict:", l1_reconstruido)
print(f"Son iguales: {l1 == l1_reconstruido}")

# Polimorfismo: mismo bucle, tipos distintos
print("\n─── Tipos polimórficos ───")
catalogo_prueba = [l1, l2, r1, d1]
for m in catalogo_prueba:
    print(f"  {m.tipo():8} → {m.titulo}")
```

**Decisiones de diseño a comentar en clase:**
- `__hash__` se define junto a `__eq__` para que los materiales puedan usarse en sets y como claves de dict.
- `from_dict` usa `cls(...)` en subclases y luego corrige `id` y `disponible` — así el `__init__` asignaría un id nuevo, pero lo sobreescribimos con el guardado. Alternativa: pasar `_override_id` como parámetro.
- `to_dict` en subclases llama a `super().to_dict()` y luego actualiza — patrón de extensión limpio.
</details>

---
## Bloque 2 — `Usuario` y `Prestamo`

**Enunciado:** Implementa dos clases de soporte:

### `Usuario`
- `__init__(nombre, email, max_prestamos=3)` — id autoincremental (`U001`…), nombre, email, lista de `prestamos_activos` (ids de material).
- `puede_pedir_prestado()` → `True` si `len(prestamos_activos) < max_prestamos`.
- `registrar_prestamo(id_material)` / `registrar_devolucion(id_material)`.
- `to_dict()` / `@classmethod from_dict(cls, datos)`.
- `__str__`, `__repr__`, `__eq__` por `id`.

### `Prestamo`
- `__init__(usuario, material, dias_plazo=14)` — guarda referencias, `fecha_prestamo=date.today()`, `fecha_devolucion=None`.
- `@property fecha_limite` → `fecha_prestamo + timedelta(dias_plazo)`.
- `@property activo` → `fecha_devolucion is None`.
- `@property dias_restantes` → días hasta `fecha_limite` (negativo si vencido).
- `@property en_mora` → `True` si activo y `dias_restantes < 0`.
- `cerrar()` → registra `fecha_devolucion = date.today()`.
- `to_dict()` — guarda ids, no los objetos completos.
- `__str__` legible con estado visual (activo / vencido / devuelto).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from datetime import date, timedelta


<details>
<summary>💡 Ver solución — Bloque 2</summary>

```python
from datetime import date, timedelta

class Usuario:
    _contador = 0

    def __init__(self, nombre, email, max_prestamos=3):
        Usuario._contador += 1
        self.id               = f"U{Usuario._contador:03d}"
        self.nombre           = nombre.strip().title()
        self.email            = email.strip().lower()
        self.max_prestamos    = int(max_prestamos)
        self.prestamos_activos = []   # lista de id_material

    def puede_pedir_prestado(self):
        return len(self.prestamos_activos) < self.max_prestamos

    def registrar_prestamo(self, id_material):
        if not self.puede_pedir_prestado():
            raise LimitePrestamosAlcanzado(
                f"{self.nombre} ya tiene {len(self.prestamos_activos)} "
                f"préstamo(s) activo(s) (máximo: {self.max_prestamos})"
            )
        if id_material in self.prestamos_activos:
            raise ErrorBiblioteca(f"{self.nombre} ya tiene prestado '{id_material}'")
        self.prestamos_activos.append(id_material)

    def registrar_devolucion(self, id_material):
        if id_material not in self.prestamos_activos:
            raise PrestamoNoEncontrado(
                f"{self.nombre} no tiene ningún préstamo activo para '{id_material}'"
            )
        self.prestamos_activos.remove(id_material)

    def to_dict(self):
        return {
            "id":                self.id,
            "nombre":            self.nombre,
            "email":             self.email,
            "max_prestamos":     self.max_prestamos,
            "prestamos_activos": self.prestamos_activos[:],
        }

    @classmethod
    def from_dict(cls, datos):
        obj = cls(datos["nombre"], datos["email"], datos.get("max_prestamos", 3))
        obj.id                = datos["id"]
        obj.prestamos_activos = datos.get("prestamos_activos", [])
        return obj

    def __str__(self):
        activos = len(self.prestamos_activos)
        return (
            f"[{self.id}] {self.nombre} ({self.email}) "
            f"— {activos}/{self.max_prestamos} préstamos"
        )

    def __repr__(self):
        return f"Usuario(id={self.id!r}, nombre={self.nombre!r})"

    def __eq__(self, otro):
        if not isinstance(otro, Usuario):
            return NotImplemented
        return self.id == otro.id

    def __hash__(self):
        return hash(self.id)


class Prestamo:
    _contador = 0

    def __init__(self, usuario, material, dias_plazo=14):
        Prestamo._contador   += 1
        self.id               = f"P{Prestamo._contador:05d}"
        self.usuario          = usuario
        self.material         = material
        self.dias_plazo       = int(dias_plazo)
        self.fecha_prestamo   = date.today()
        self.fecha_devolucion = None

    @property
    def fecha_limite(self):
        return self.fecha_prestamo + timedelta(days=self.dias_plazo)

    @property
    def activo(self):
        return self.fecha_devolucion is None

    @property
    def dias_restantes(self):
        if not self.activo:
            return 0
        return (self.fecha_limite - date.today()).days

    @property
    def en_mora(self):
        return self.activo and self.dias_restantes < 0

    def cerrar(self):
        self.fecha_devolucion = date.today()

    def to_dict(self):
        return {
            "id":               self.id,
            "id_usuario":       self.usuario.id,
            "id_material":      self.material.id,
            "dias_plazo":       self.dias_plazo,
            "fecha_prestamo":   str(self.fecha_prestamo),
            "fecha_devolucion": str(self.fecha_devolucion) if self.fecha_devolucion else None,
        }

    def __str__(self):
        if not self.activo:
            estado = f"✅ Devuelto {self.fecha_devolucion}"
        elif self.en_mora:
            estado = f"🔴 VENCIDO ({abs(self.dias_restantes)} días de mora)"
        else:
            estado = f"📖 Activo ({self.dias_restantes} días restantes)"
        return (
            f"[{self.id}] {self.usuario.nombre} ← '{self.material.titulo}' "
            f"| {self.fecha_prestamo} → {self.fecha_limite} | {estado}"
        )


# ─── Pruebas ─────────────────────────────────────────────
Material._contador = 0
Usuario._contador  = 0
Prestamo._contador = 0

u1 = Usuario("ana garcía", "ana@mail.com", max_prestamos=2)
u2 = Usuario("Luis Pérez", "luis@mail.com")
print(u1)
print(u2)

l1 = Libro("Dune", "Frank Herbert", "978-84-450-7652-0", "CF", 604)

p = Prestamo(u1, l1, dias_plazo=14)
print(f"\n{p}")
print(f"Fecha límite: {p.fecha_limite}")
print(f"Días restantes: {p.dias_restantes}")

u1.registrar_prestamo(l1.id)
print(f"\nUsuario tras prestar: {u1}")

# Simular mora
p.fecha_prestamo = date.today() - timedelta(days=20)
print(f"\nSimulando mora: {p}")
print(f"En mora: {p.en_mora}")

# Devolver
p.cerrar()
u1.registrar_devolucion(l1.id)
print(f"\nTras devolver: {p}")
print(f"Usuario: {u1}")
```

**Decisiones de diseño:**
- `Prestamo` guarda **referencias** al objeto `usuario` y `material`, no solo sus ids — así en tiempo de ejecución podemos acceder a todos sus datos sin búsquedas adicionales. `to_dict()` guarda solo los ids para serialización.
- `dias_restantes` devuelve 0 si el préstamo ya está cerrado — evita valores negativos confusos en préstamos ya devueltos.
- `prestamos_activos` en `Usuario` es una lista de `id_material` (strings), no de objetos `Material` — más ligero y fácil de serializar.
</details>

---
## Bloque 3 — `Biblioteca`: catálogo y usuarios

**Enunciado:** Implementa la clase `Biblioteca` con la gestión del catálogo y los usuarios. Aún no los préstamos (eso es el bloque 4).

**Catálogo:**
- `añadir_material(material)` — lanza `DatosInvalidos` si no es instancia de `Material`.
- `eliminar_material(id_material)` — lanza `MaterialNoEncontrado` si no existe o `MaterialNoDisponible` si está prestado.
- `buscar_por_id(id_material)` → material o lanza `MaterialNoEncontrado`.
- `buscar(termino)` → lista de materiales cuyo título, autor/director/editorial contiene el término (insensible a mayúsculas).
- `filtrar(tipo=None, disponible=None)` → lista filtrada por tipo (`"Libro"`, `"Revista"`, `"DVD"`) y/o disponibilidad.

**Usuarios:**
- `registrar_usuario(usuario)` — lanza `DatosInvalidos` si el email ya existe.
- `buscar_usuario(id_usuario)` → usuario o lanza `UsuarioNoEncontrado`.

**Colecciones internas:** usa diccionarios `{id: objeto}` para `O(1)` en búsquedas por id.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Bloque 3</summary>

```python
class Biblioteca:
    def __init__(self, nombre):
        self.nombre     = nombre
        self._catalogo  = {}   # {id_material: Material}
        self._usuarios  = {}   # {id_usuario: Usuario}
        self._prestamos = {}   # {id_prestamo: Prestamo}  — bloque 4

    # ─── Catálogo ─────────────────────────────────────────
    def añadir_material(self, material):
        if not isinstance(material, Material):
            raise DatosInvalidos(f"Se esperaba un Material, no {type(material).__name__}")
        self._catalogo[material.id] = material
        print(f"📚 Añadido: {material}")

    def eliminar_material(self, id_material):
        material = self.buscar_por_id(id_material)   # lanza MaterialNoEncontrado
        if not material.disponible:
            raise MaterialNoDisponible(
                f"No se puede eliminar '{material.titulo}': está prestado"
            )
        del self._catalogo[id_material]
        print(f"🗑️  Eliminado: {material.titulo}")
        return material

    def buscar_por_id(self, id_material):
        material = self._catalogo.get(id_material)
        if material is None:
            raise MaterialNoEncontrado(f"No existe ningún material con id '{id_material}'")
        return material

    def buscar(self, termino):
        """Búsqueda en título y campos de texto de cada tipo."""
        termino = termino.strip().lower()
        resultados = []
        for m in self._catalogo.values():
            campos = [m.titulo]
            if isinstance(m, Libro):
                campos += [m.autor, m.isbn, m.genero]
            elif isinstance(m, Revista):
                campos += [m.editorial]
            elif isinstance(m, DVD):
                campos += [m.director, m.clasificacion]
            if any(termino in campo.lower() for campo in campos):
                resultados.append(m)
        return resultados

    def filtrar(self, tipo=None, disponible=None):
        """Filtra el catálogo por tipo y/o disponibilidad."""
        resultado = list(self._catalogo.values())
        if tipo is not None:
            resultado = [m for m in resultado if m.tipo() == tipo]
        if disponible is not None:
            resultado = [m for m in resultado if m.disponible == disponible]
        return resultado

    # ─── Usuarios ─────────────────────────────────────────
    def registrar_usuario(self, usuario):
        if not isinstance(usuario, Usuario):
            raise DatosInvalidos(f"Se esperaba un Usuario, no {type(usuario).__name__}")
        emails_existentes = {u.email for u in self._usuarios.values()}
        if usuario.email in emails_existentes:
            raise DatosInvalidos(f"El email '{usuario.email}' ya está registrado")
        self._usuarios[usuario.id] = usuario
        print(f"👤 Registrado: {usuario}")

    def buscar_usuario(self, id_usuario):
        usuario = self._usuarios.get(id_usuario)
        if usuario is None:
            raise UsuarioNoEncontrado(f"No existe ningún usuario con id '{id_usuario}'")
        return usuario

    # ─── Utilidades básicas ───────────────────────────────
    def __len__(self):
        return len(self._catalogo)

    def __str__(self):
        return (
            f"{self.nombre} — {len(self._catalogo)} materiales, "
            f"{len(self._usuarios)} usuarios, "
            f"{len(self._prestamos)} préstamos activos"
        )


# ─── Pruebas ─────────────────────────────────────────────
Material._contador = 0
Usuario._contador  = 0

bib = Biblioteca("Biblioteca Municipal")

libros = [
    Libro("El Quijote",          "Miguel de Cervantes", "978-84-376-0494-7", "Clásico",         863),
    Libro("Dune",                "Frank Herbert",       "978-84-450-7652-0", "Ciencia ficción", 604),
    Libro("Sapiens",             "Yuval Noah Harari",   "978-84-9992-950-3", "Ensayo",          443),
    Libro("Python para todos",   "Charles Severance",   "978-1-530051-43-7", "Tecnología",      242),
]
revistas = [
    Revista("National Geographic", "NGS",         312, "Mayo 2025"),
    Revista("MIT Tech Review",     "MIT Press",    89,  "Junio 2025"),
]
dvds = [
    DVD("Blade Runner 2049",  "Denis Villeneuve", 164, "16"),
    DVD("Interstellar",       "Christopher Nolan", 169, "TP"),
]

for m in libros + revistas + dvds:
    bib.añadir_material(m)

for nombre, email in [("Ana García", "ana@mail.com"), ("Luis Pérez", "luis@mail.com"),
                      ("Marta López", "marta@mail.com")]:
    bib.registrar_usuario(Usuario(nombre, email))

print(f"\n{bib}")

print("\n─── Búsqueda: 'dune' ───")
for m in bib.buscar("dune"):
    print(f"  {m}")

print("\n─── Filtro: Libros disponibles ───")
for m in bib.filtrar(tipo="Libro", disponible=True):
    print(f"  {m}")
```

**Decisiones de diseño:**
- Diccionarios internos (`_catalogo`, `_usuarios`, `_prestamos`) con id como clave — búsqueda en O(1).
- `buscar` usa `isinstance` dentro de la propia clase — es aceptable porque `Biblioteca` es la capa que conoce todas las subclases.
- `{u.email for u in self._usuarios.values()}` — set comprehension para comprobación de duplicados en O(1).
</details>

---
## Bloque 4 — Préstamos, devoluciones y búsqueda polimórfica

**Enunciado:** Completa la clase `Biblioteca` añadiendo la gestión de préstamos:

- **`prestar(id_usuario, id_material, dias_plazo=14)`**:
  1. Busca el usuario y el material (lanza si no existen).
  2. Verifica que el usuario puede pedir prestado.
  3. Llama a `material.prestar()` (polimorfismo — funciona para Libro, Revista o DVD).
  4. Llama a `usuario.registrar_prestamo(id_material)`.
  5. Crea el `Prestamo`, lo guarda y lo devuelve.

- **`devolver(id_usuario, id_material)`**:
  1. Encuentra el préstamo activo correspondiente.
  2. Cierra el préstamo, devuelve el material y actualiza el usuario.

- **`prestamos_activos()`** → lista de préstamos sin `fecha_devolucion`.
- **`prestamos_en_mora()`** → lista de préstamos activos con `en_mora == True`.
- **`historial_usuario(id_usuario)`** → todos los préstamos de ese usuario.
- **`listar_catalogo()`** → imprime el catálogo completo organizado por tipo.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — añadir métodos a la clase Biblioteca


<details>
<summary>💡 Ver solución — Bloque 4</summary>

```python
# Redefinir Biblioteca completa con todos los bloques anteriores más los nuevos métodos

class Biblioteca:
    def __init__(self, nombre):
        self.nombre     = nombre
        self._catalogo  = {}
        self._usuarios  = {}
        self._prestamos = {}

    # ── Catálogo (Bloque 3) ───────────────────────────────
    def añadir_material(self, material):
        if not isinstance(material, Material):
            raise DatosInvalidos(f"Se esperaba un Material")
        self._catalogo[material.id] = material

    def eliminar_material(self, id_material):
        material = self.buscar_por_id(id_material)
        if not material.disponible:
            raise MaterialNoDisponible(f"No se puede eliminar '{material.titulo}': está prestado")
        del self._catalogo[id_material]
        return material

    def buscar_por_id(self, id_material):
        m = self._catalogo.get(id_material)
        if m is None:
            raise MaterialNoEncontrado(f"Material '{id_material}' no encontrado")
        return m

    def buscar(self, termino):
        termino = termino.strip().lower()
        resultados = []
        for m in self._catalogo.values():
            campos = [m.titulo]
            if isinstance(m, Libro):   campos += [m.autor, m.isbn, m.genero]
            elif isinstance(m, Revista): campos += [m.editorial]
            elif isinstance(m, DVD):   campos += [m.director]
            if any(termino in c.lower() for c in campos):
                resultados.append(m)
        return resultados

    def filtrar(self, tipo=None, disponible=None):
        r = list(self._catalogo.values())
        if tipo       is not None: r = [m for m in r if m.tipo() == tipo]
        if disponible is not None: r = [m for m in r if m.disponible == disponible]
        return r

    # ── Usuarios (Bloque 3) ───────────────────────────────
    def registrar_usuario(self, usuario):
        if not isinstance(usuario, Usuario):
            raise DatosInvalidos("Se esperaba un Usuario")
        if any(u.email == usuario.email for u in self._usuarios.values()):
            raise DatosInvalidos(f"El email '{usuario.email}' ya está registrado")
        self._usuarios[usuario.id] = usuario

    def buscar_usuario(self, id_usuario):
        u = self._usuarios.get(id_usuario)
        if u is None:
            raise UsuarioNoEncontrado(f"Usuario '{id_usuario}' no encontrado")
        return u

    # ── Préstamos (Bloque 4) ──────────────────────────────
    def prestar(self, id_usuario, id_material, dias_plazo=14):
        usuario  = self.buscar_usuario(id_usuario)
        material = self.buscar_por_id(id_material)

        if not usuario.puede_pedir_prestado():
            raise LimitePrestamosAlcanzado(
                f"{usuario.nombre} ha alcanzado el límite de {usuario.max_prestamos} préstamos"
            )

        material.prestar()                         # polimorfismo — funciona igual para Libro, Revista, DVD
        usuario.registrar_prestamo(material.id)

        prestamo = Prestamo(usuario, material, dias_plazo)
        self._prestamos[prestamo.id] = prestamo

        print(f"📖 Préstamo registrado: {prestamo}")
        return prestamo

    def devolver(self, id_usuario, id_material):
        usuario  = self.buscar_usuario(id_usuario)
        material = self.buscar_por_id(id_material)

        # Buscar el préstamo activo correspondiente
        prestamo = None
        for p in self._prestamos.values():
            if p.activo and p.usuario.id == id_usuario and p.material.id == id_material:
                prestamo = p
                break

        if prestamo is None:
            raise PrestamoNoEncontrado(
                f"No hay préstamo activo de '{material.titulo}' para {usuario.nombre}"
            )

        prestamo.cerrar()
        material.devolver()
        usuario.registrar_devolucion(material.id)

        mora = "⚠️  (con mora)" if prestamo.dias_restantes < 0 else ""
        print(f"✅ Devolución: '{material.titulo}' devuelto por {usuario.nombre} {mora}")
        return prestamo

    def prestamos_activos(self):
        return [p for p in self._prestamos.values() if p.activo]

    def prestamos_en_mora(self):
        return [p for p in self._prestamos.values() if p.en_mora]

    def historial_usuario(self, id_usuario):
        usuario = self.buscar_usuario(id_usuario)
        return [p for p in self._prestamos.values() if p.usuario.id == id_usuario]

    def listar_catalogo(self):
        from collections import defaultdict
        grupos = defaultdict(list)
        for m in self._catalogo.values():
            grupos[m.tipo()].append(m)

        print(f"\n{'═'*70}")
        print(f"  CATÁLOGO — {self.nombre} ({len(self._catalogo)} materiales)")
        print(f"{'═'*70}")
        for tipo in sorted(grupos):
            grupo = grupos[tipo]
            disp  = sum(1 for m in grupo if m.disponible)
            print(f"\n  {tipo.upper()} ({len(grupo)} — {disp} disponibles)")
            print(f"  {'─'*68}")
            for m in grupo:
                print(f"  {m}")
        print(f"{'═'*70}")

    # ── Dunder ────────────────────────────────────────────
    def __len__(self):
        return len(self._catalogo)

    def __str__(self):
        activos = len(self.prestamos_activos())
        return (
            f"{self.nombre} — {len(self._catalogo)} materiales, "
            f"{len(self._usuarios)} usuarios, {activos} préstamos activos"
        )


# ─── Escenario de prueba completo ────────────────────────
Material._contador = 0
Usuario._contador  = 0
Prestamo._contador = 0

bib = Biblioteca("Biblioteca Municipal")

for m in [
    Libro("El Quijote",        "Miguel de Cervantes", "978-84-376-0494-7", "Clásico",         863),
    Libro("Dune",              "Frank Herbert",       "978-84-450-7652-0", "Ciencia ficción", 604),
    Libro("Sapiens",           "Yuval Noah Harari",   "978-84-9992-950-3", "Ensayo",          443),
    Libro("Python para todos", "Charles Severance",   "978-1-530051-43-7", "Tecnología",      242),
    Revista("National Geographic", "NGS",      312, "Mayo 2025"),
    Revista("MIT Tech Review",     "MIT Press",  89, "Junio 2025"),
    DVD("Blade Runner 2049", "Denis Villeneuve",  164, "16"),
    DVD("Interstellar",      "Christopher Nolan", 169, "TP"),
]:
    bib.añadir_material(m)

u1 = Usuario("Ana García",  "ana@mail.com",   max_prestamos=3)
u2 = Usuario("Luis Pérez",  "luis@mail.com")
u3 = Usuario("Marta López", "marta@mail.com")
for u in [u1, u2, u3]:
    bib.registrar_usuario(u)

bib.listar_catalogo()

print("\n─── Préstamos ───")
p1 = bib.prestar(u1.id, "M001", dias_plazo=14)
p2 = bib.prestar(u1.id, "M005", dias_plazo=7)
p3 = bib.prestar(u2.id, "M007", dias_plazo=3)

# Simular mora en p3
p3.fecha_prestamo = date.today() - timedelta(days=10)

print("\n─── Activos ───")
for p in bib.prestamos_activos():
    print(f"  {p}")

print("\n─── En mora ───")
for p in bib.prestamos_en_mora():
    print(f"  {p}")

print("\n─── Devolucion ───")
bib.devolver(u1.id, "M001")

print("\n─── Historial de Ana ───")
for p in bib.historial_usuario(u1.id):
    print(f"  {p}")
```

**Decisiones de diseño:**
- `material.prestar()` en `Biblioteca.prestar` — polimorfismo puro: si mañana añadimos `Audiolibro(Material)`, el método `prestar` de `Biblioteca` no cambia.
- Buscar el préstamo activo con un bucle es O(n) — para una biblioteca pequeña es correcto. En producción, añadiríamos un índice secundario `{(id_usuario, id_material): id_prestamo}`.
- `prestamo.cerrar()` + `material.devolver()` + `usuario.registrar_devolucion()` — tres operaciones atómicas. Si alguna falla a mitad, el estado queda inconsistente. En un sistema real usaríamos transacciones.
</details>

---
## Bloque 5 (reto) — Persistencia, informes y menú interactivo

**Enunciado:** Completa el sistema con tres módulos finales:

### 5.1 Persistencia JSON
Añade a `Biblioteca`:
- `guardar(ruta)` — serializa catálogo, usuarios y préstamos a un fichero JSON.
- `@classmethod cargar(cls, ruta)` — reconstruye la `Biblioteca` desde el JSON.

Para reconstruir materiales desde JSON necesitas un **registro de clases** (`CLASES_MATERIAL = {"Libro": Libro, "Revista": Revista, "DVD": DVD}`). Úsalo en `cargar`.

### 5.2 Informes
Añade a `Biblioteca`:
- `informe_general()` — estadísticas globales: total de materiales por tipo, % de disponibilidad, top 3 materiales más pedidos (por historial), usuario con más préstamos activos.

### 5.3 Menú interactivo
Escribe una función `menu(biblioteca)` que muestre un menú en bucle `while True` con opciones:
```
1. Buscar material
2. Pedir prestado
3. Devolver
4. Ver catálogo
5. Ver préstamos activos
6. Informe general
7. Guardar y salir
```
Usa `try/except ErrorBiblioteca` para capturar errores sin romper el bucle.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import json
from pathlib import Path
from collections import Counter


<details>
<summary>💡 Ver solución — Bloque 5</summary>

```python
import json
from pathlib import Path
from collections import Counter

# Registro de clases para deserialización
CLASES_MATERIAL = {
    "Libro":   Libro,
    "Revista": Revista,
    "DVD":     DVD,
}


# ── 5.1 Persistencia ──────────────────────────────────────

def guardar(self, ruta):
    """Serializa todo el estado de la biblioteca a JSON."""
    datos = {
        "nombre":    self.nombre,
        "catalogo":  [m.to_dict() for m in self._catalogo.values()],
        "usuarios":  [u.to_dict() for u in self._usuarios.values()],
        "prestamos": [p.to_dict() for p in self._prestamos.values()],
        "_contadores": {
            "material": Material._contador,
            "usuario":  Usuario._contador,
            "prestamo": Prestamo._contador,
        }
    }
    Path(ruta).write_text(
        json.dumps(datos, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"💾 Biblioteca guardada en '{ruta}'")


@classmethod
def cargar(cls, ruta):
    """Reconstruye la Biblioteca desde un JSON."""
    ruta_path = Path(ruta)
    if not ruta_path.exists():
        raise FileNotFoundError(f"No se encuentra el archivo '{ruta}'")

    datos = json.loads(ruta_path.read_text(encoding="utf-8"))

    # Restaurar contadores
    contadores = datos.get("_contadores", {})
    Material._contador = contadores.get("material", 0)
    Usuario._contador  = contadores.get("usuario",  0)
    Prestamo._contador = contadores.get("prestamo", 0)

    bib = cls(datos["nombre"])

    # Reconstruir catálogo
    for d in datos["catalogo"]:
        ClaseM = CLASES_MATERIAL.get(d["clase"])
        if ClaseM:
            bib._catalogo[d["id"]] = ClaseM.from_dict(d)

    # Reconstruir usuarios
    for d in datos["usuarios"]:
        bib._usuarios[d["id"]] = Usuario.from_dict(d)

    # Reconstruir préstamos (necesitan referencias a objetos ya cargados)
    for d in datos["prestamos"]:
        usuario  = bib._usuarios.get(d["id_usuario"])
        material = bib._catalogo.get(d["id_material"])
        if usuario and material:
            p = Prestamo.__new__(Prestamo)
            p.id               = d["id"]
            p.usuario          = usuario
            p.material         = material
            p.dias_plazo       = d["dias_plazo"]
            p.fecha_prestamo   = date.fromisoformat(d["fecha_prestamo"])
            p.fecha_devolucion = (
                date.fromisoformat(d["fecha_devolucion"])
                if d["fecha_devolucion"] else None
            )
            bib._prestamos[p.id] = p

    print(f"📂 Biblioteca cargada desde '{ruta}'")
    return bib


# ── 5.2 Informes ──────────────────────────────────────────

def informe_general(self):
    from collections import Counter, defaultdict

    print(f"\n{'═'*70}")
    print(f"  INFORME GENERAL — {self.nombre}")
    print(f"{'═'*70}")

    # Materiales por tipo
    grupos = defaultdict(list)
    for m in self._catalogo.values():
        grupos[m.tipo()].append(m)

    print("\n  CATÁLOGO:")
    total = len(self._catalogo)
    for tipo, grupo in sorted(grupos.items()):
        disp = sum(1 for m in grupo if m.disponible)
        pct  = (disp / len(grupo) * 100) if grupo else 0
        print(f"    {tipo:<12} {len(grupo):>3} uds   disponibles: {disp} ({pct:.0f}%)")
    print(f"    {'TOTAL':<12} {total:>3} uds")

    # Top 3 materiales más prestados
    conteo = Counter(p.material.id for p in self._prestamos.values())
    print("\n  TOP 3 MATERIALES MÁS PRESTADOS:")
    for id_m, veces in conteo.most_common(3):
        m = self._catalogo.get(id_m)
        if m:
            print(f"    {veces:>2}x — {m.titulo} ({m.tipo()})")

    # Usuarios
    print("\n  USUARIOS:")
    print(f"    Registrados: {len(self._usuarios)}")
    activos_por_usuario = {u.id: len(u.prestamos_activos) for u in self._usuarios.values()}
    if activos_por_usuario:
        top_id = max(activos_por_usuario, key=activos_por_usuario.get)
        top_u  = self._usuarios[top_id]
        print(f"    Con más préstamos activos: {top_u.nombre} ({activos_por_usuario[top_id]})")

    # Mora
    mora = self.prestamos_en_mora()
    print(f"\n  PRÉSTAMOS:")
    print(f"    Total registrados: {len(self._prestamos)}")
    print(f"    Activos:           {len(self.prestamos_activos())}")
    print(f"    En mora:           {len(mora)}")
    if mora:
        for p in mora:
            print(f"      ⚠️  {p}")

    print(f"{'═'*70}")


# ── 5.3 Menú interactivo ──────────────────────────────────

def menu(biblioteca):
    """Menú interactivo de la biblioteca."""
    OPCIONES = {
        "1": "Buscar material",
        "2": "Pedir prestado",
        "3": "Devolver",
        "4": "Ver catálogo",
        "5": "Ver préstamos activos",
        "6": "Informe general",
        "7": "Guardar y salir",
    }

    while True:
        print(f"\n{'─'*40}")
        print(f"  {biblioteca.nombre}")
        print(f"{'─'*40}")
        for clave, desc in OPCIONES.items():
            print(f"  {clave}. {desc}")
        print(f"{'─'*40}")

        opcion = input("Elige una opción: ").strip()

        try:
            if opcion == "1":
                termino = input("Término de búsqueda: ").strip()
                resultados = biblioteca.buscar(termino)
                if resultados:
                    for m in resultados: print(f"  {m}")
                else:
                    print("  Sin resultados.")

            elif opcion == "2":
                id_u = input("ID usuario: ").strip()
                id_m = input("ID material: ").strip()
                biblioteca.prestar(id_u, id_m)

            elif opcion == "3":
                id_u = input("ID usuario: ").strip()
                id_m = input("ID material: ").strip()
                biblioteca.devolver(id_u, id_m)

            elif opcion == "4":
                biblioteca.listar_catalogo()

            elif opcion == "5":
                activos = biblioteca.prestamos_activos()
                if activos:
                    for p in activos: print(f"  {p}")
                else:
                    print("  No hay préstamos activos.")

            elif opcion == "6":
                biblioteca.informe_general()

            elif opcion == "7":
                ruta = input("Ruta del archivo (Enter = biblioteca.json): ").strip()
                if not ruta: ruta = "biblioteca.json"
                biblioteca.guardar(ruta)
                print("Hasta pronto.")
                break

            else:
                print("Opción no válida. Elige entre 1 y 7.")

        except ErrorBiblioteca as e:
            print(f"⚠️  {type(e).__name__}: {e}")
        except KeyboardInterrupt:
            print("\nSaliendo sin guardar.")
            break


# Inyectar los métodos en la clase (monkey-patching — para esta demo)
Biblioteca.guardar        = guardar
Biblioteca.cargar         = cargar
Biblioteca.informe_general = informe_general


# ── Prueba de persistencia ────────────────────────────────
# (Usa el estado del bloque anterior)
bib.guardar("biblioteca.json")

Material._contador = 0
Usuario._contador  = 0
Prestamo._contador = 0

bib2 = Biblioteca.cargar("biblioteca.json")
print(bib2)

bib2.informe_general()

# Descomentar para probar el menú en modo interactivo:
# menu(bib2)
```

**Decisiones de diseño:**
- `CLASES_MATERIAL` es el patrón **registro** — al cargar el JSON miramos la clave `"clase"` y la usamos para saber qué `from_dict` llamar. Añadir un nuevo tipo solo requiere añadir una entrada al diccionario.
- Los contadores se guardan en el JSON para evitar colisiones de id al recargar. Sin esto, cargar y crear nuevos materiales podría generar ids duplicados.
- El menú captura `ErrorBiblioteca` pero deja pasar `KeyboardInterrupt` — así Ctrl+C sale limpiamente.
- `monkey-patching` (inyectar métodos con `Biblioteca.guardar = guardar`) es solo para esta demo de notebook. En producción todos los métodos irían dentro de la definición de la clase.
</details>

---
## Reflexión final — Decisiones de diseño del proyecto

Ahora que el sistema está completo, repasa estas preguntas. No hay respuestas únicas — el objetivo es que puedas **argumentar** tu elección.

### Preguntas para debatir en clase

**1. Herencia vs composición**
> `Revista` hereda de `Material`. Pero, ¿tendría más sentido que `Material` tuviera un atributo `tipo_info` que fuera un objeto separado?

**2. Dónde poner la lógica**
> `material.prestar()` está en `Material`. `Biblioteca.prestar()` orquesta la operación completa. ¿Por qué no meter todo en `Material`? ¿Por qué no meter todo en `Biblioteca`?

**3. Relaciones entre objetos**
> `Prestamo` guarda referencias a `Usuario` y `Material` en memoria, pero solo ids en JSON. ¿Cuándo es mejor guardar el objeto completo y cuándo solo el id?

**4. Consistencia del estado**
> En `devolver()` hacemos tres operaciones: `prestamo.cerrar()`, `material.devolver()`, `usuario.registrar_devolucion()`. Si la segunda falla, el estado queda a medio actualizar. ¿Cómo lo resolverías?

**5. Escalar el sistema**
> Si la biblioteca tiene 50.000 materiales, buscar el préstamo activo con un bucle es demasiado lento. ¿Qué estructura de datos añadirías?

---

## Extensiones opcionales

Si quieres seguir practicando, aquí tienes ideas ordenadas de menor a mayor dificultad:

| # | Extensión | Conceptos que refuerza |
|---|-----------|------------------------|
| A | Añadir `Audiolibro(Material)` con `narrador` y `duracion_min` | Herencia, `from_dict` |
| B | `reservar(id_usuario, id_material)` — cola de espera si está prestado | Colecciones (deque), POO |
| C | Calcular multa por mora (€/día) y añadirla al informe | Propiedades, aritmética de fechas |
| D | Exportar el informe a `.txt` y el catálogo a `.csv` | Archivos (C8) |
| E | Validar el formato del email con expresiones regulares (`re`) | Módulo `re` |
| F | Pruebas unitarias con `unittest` para `prestar` y `devolver` | Testing básico |

---

## Cierre del Módulo 1

### Lo que has construido en este proyecto

```
Jerarquía de excepciones propias
Clase abstracta Material con @abstractmethod
Tres subclases concretas (Libro, Revista, DVD)
Serialización to_dict / from_dict en cada nivel
Clases de soporte (Usuario, Prestamo) con propiedades calculadas
Orquestador Biblioteca con búsquedas, filtros y estadísticas
Persistencia completa en JSON (guardar y cargar)
Menú interactivo con manejo de errores robusto
```

### Lo que has aplicado del Módulo 1

```
Variables · Tipos · Operadores       → En todas partes
Control de flujo · Bucles            → Menú, búsquedas, filtros
Listas · Diccionarios · Sets         → Catálogo, usuarios, historial
Funciones · *args · **kwargs         → Métodos de búsqueda, informes
Archivos (JSON)                      → guardar() / cargar()
Excepciones personalizadas           → Dominio bancario completo
POO: ABC, herencia, polimorfismo     → Toda la jerarquía Material
POO: encapsulación, @property        → Saldo, fechas, estados
```

---

## ¿Qué viene en el Módulo 2?

El **Módulo 2** da el salto de Python puro a las herramientas que usan los profesionales de datos e IA:

| Clase | Tema |
|-------|------|
| M2 C1 | NumPy — operaciones vectoriales y matrices |
| M2 C2 | Pandas — DataFrames y análisis de datos |
| M2 C3 | Visualización con Matplotlib y Seaborn |
| M2 C4 | Introducción al Machine Learning con scikit-learn |

Todo lo que has aprendido en el Módulo 1 es la base. A partir de aquí, usarás clases, excepciones, archivos y colecciones continuamente — pero sobre librerías construidas por otros.